# Phase 4b, second addendum: K2 on the students and the base-model assay (A100 80 GB)

Reads `results/phase4b/k2_scores.csv`, `k2_vs_sham.csv` and `base_assay.csv` (written by `python -m zoo.phase4b_followup`). Trains nothing. Writes `reports/figures/phase4b_k2_base.png`. Committed without outputs.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
S = pd.read_csv(ROOT / "results/phase4b/k2_scores.csv")
C = pd.read_csv(ROOT / "results/phase4b/k2_vs_sham.csv")
B = pd.read_csv(ROOT / "results/phase4b/base_assay.csv")

In [ ]:
INK, MUTED, GRID = "#222222", "#6b6b6b", "#e6e6e6"
COL = {"student": "#c0504d", "sham student": "#4f6d8f", "1x-teacher student": "#d08a1f", "naive": "#9aa4b1", "base": "#2e7d5b"}
fig, (A, P) = plt.subplots(1, 2, figsize=(11.5, 4.4), constrained_layout=True)
doses = ["01x", "08x", "16x", "32x", "64x"]
jit = {0: -0.12, 1: -0.06, 2: 0.0, 3: 0.06, 4: 0.12}
for arm, off in (("sham student", -0.18), ("student", 0.18)):
    g = S[S.arm == arm]
    for i, d in enumerate(doses):
        x = g[g.dose == d]
        A.scatter([i + off + jit[k] for k in x.seed], x.K2, s=22, color=COL[arm], edgecolor="white", linewidth=0.5, zorder=3,
                  label={"student": "active-teacher students", "sham student": "sham students"}[arm] if i == 0 else None)
x = S[S.arm == "1x-teacher student"]
A.scatter([5 + jit[k] for k in x.seed], x.K2, s=22, color=COL["1x-teacher student"], edgecolor="white", linewidth=0.5,
          zorder=3, label="1x-teacher students (64x)")
nv = S[S.label == "naive"]
A.axhspan(nv.K2.min(), nv.K2.max(), color=GRID, zorder=0)
A.text(5.45, nv.K2.max() + 0.005, "naive models", fontsize=7.5, color=MUTED, ha="right", va="bottom")
A.set_xticks(range(6), [d.lstrip("0") for d in doses] + ["1x-teacher\n64x"])
A.set_xlabel("distillation dose"); A.set_ylabel("K2 (trigger-vs-decoy AUROC, sites 3-7)")
A.set_ylim(0.55, 1.01); A.legend(frameon=False, fontsize=8, loc="upper left")
A.set_title("A  K2: probe of the active model of the same seed, held-out prompts", loc="left", fontsize=10)
A.text(0.99, 0.02, f"above its sham = dK2 > R = {C.R.iloc[0]:.3f}", transform=A.transAxes, ha="right", va="bottom",
       fontsize=7.5, color=MUTED)

med = lambda s: np.median([int(v) for v in s.split("/")])
groups = [("naive", "t_naive"), ("sham 64x", "t_sham_64x"), ("sham 8x", "t_sham_08x"), ("sham 1x", "t_sham_01x"), ("base model", "t_base")]
for i, (lab, col) in enumerate(groups):
    y = [med(v) for v in B[col]]
    c = COL["naive"] if lab == "naive" else (COL["base"] if lab == "base model" else COL["sham student"])
    P.scatter([i + jit[k] for k in B.seed], y, s=24, color=c, edgecolor="white", linewidth=0.5, zorder=3)
    P.hlines(np.median(y), i - 0.25, i + 0.25, color=INK, lw=1.4, zorder=4)
P.set_xticks(range(len(groups)), [g[0] for g in groups]); P.set_ylim(0, 800)
P.set_ylabel("crossing step (median of 3 assay seeds)")
P.set_title("B  Relearning: base model against naive models and sham students", loc="left", fontsize=10)
P.text(0.99, 0.02, "one base model, plotted against each seed; bar = median", transform=P.transAxes, ha="right",
       va="bottom", fontsize=7.5, color=MUTED)
for ax in (A, P):
    ax.spines[["top", "right"]].set_visible(False)
fig.savefig(ROOT / "reports/figures/phase4b_k2_base.png", dpi=160)